> **Historical notebook — not the current data pipeline.**
>
> This predates the TMDB-compliance rewrite and downloads/processes a Kaggle-mirrored
> TMDB dataset, which the app no longer uses — TMDB's API terms don't allow redistributing
> their data, so a static Kaggle mirror isn't a compliant source either. The app now fetches
> its dataset directly from the TMDB API via `scripts/build_dataset.py` (see the README).
> Kept here only for the exploratory/portfolio history of how the recommendation approach
> was originally prototyped.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib.pyplot as plt

In [ ]:
!kaggle datasets download asaniczka/tmdb-movies-dataset-2023-930k-movies -p ../data/raw

In [ ]:
import zipfile
dataset_dir="../data/raw/tmdb-movies-dataset-2023-930k-movies.zip"
extract_to="../data/raw"
with zipfile.ZipFile(dataset_dir, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

In [ ]:
df=pd.read_csv("../data/raw/TMDB_movie_dataset_v11.csv")

In [ ]:
df.head()

In [ ]:
movies_db=df[df["popularity"]>20]
movies_db=movies_db[movies_db["adult"]==False]
movies_db=movies_db[movies_db['spoken_languages'].str.contains("English")==True]
movies_db=movies_db[movies_db['vote_average']!=0]
movies_db=movies_db[['title','vote_average','vote_count','status','release_date','runtime','overview','poster_path', 'genres','backdrop_path']]
movies_db.reset_index(inplace=True)
movies_db.shape

In [ ]:
movies_db.head()

In [ ]:
movies_db.to_csv('../data/processed/TMDB.csv', index=False) 

In [ ]:
movies_db.head()

In [ ]:
list(movies_db.columns.values)

In [ ]:
movies_db['content'] = movies_db['overview'] + ' ' + movies_db['genres'] + ' ' + movies_db['keywords']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
movies_db['content'] = movies_db['content'].fillna('')

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(movies_db['content'])
tfidf_array = tfidf_matrix.toarray()  # Convert sparse matrix to numpy array

In [ ]:
import pickle
with open('../models/content.pkl', 'wb') as file:
    pickle.dump(tfidf_array, file)

In [ ]:
user_history = [0]  # Movie IDs that the user has watched
chosen_movies = movies_db.loc[user_history]
chosen_movies.head()

In [ ]:
user_profile = np.mean(tfidf_array[user_history], axis=0)  # Average vector

In [ ]:
similarity_scores = cosine_similarity([user_profile], tfidf_array).flatten()

In [ ]:
movies_db['similarity_score'] = similarity_scores
recommended_movies = movies_db.sort_values(by='similarity_score', ascending=False)

In [ ]:
recommended_movies